# Sinhala VITS TTS — Full Google Colab Guide

This notebook trains a **multi-speaker VITS** model (Coqui TTS) on the cleaned OpenSLR Resource 30 Sinhala dataset.

## What you need

1. A **GPU** Colab runtime (`Runtime → Change runtime type → T4 / L4 / A100`).
2. Google Drive space (~3–8 GB for data + checkpoints).
3. Either:
   - the prepared dataset folder from your PC (`data/prepared/sinhala_vits`), **or**
   - cleaned WAVs + `si_lk.lines.txt` so this notebook can prepare them.

## Expected result

A checkpoint that can synthesize Sinhala speech for any of the 12 OpenSLR speakers, for example:

`ආයුබෝවන්` spoken like speaker `sin_2241`.

## Important data note

Official `si_lk.lines.txt` currently has **1251** transcript rows while the cleaned folder has **2061** WAVs. Training uses only **matched IDs** (~1248). Unmatched audio is reported and skipped; no fake transcripts are invented.

## Step 0 — Choose GPU and mount Drive

Confirm CUDA is available before installing heavy packages.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Enable a GPU runtime: Runtime → Change runtime type → GPU"
print("CUDA OK:", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/sinhala-vits")
DATASET_DIR = DRIVE_ROOT / "dataset"          # prepared VITS dataset
OUTPUT_DIR = DRIVE_ROOT / "outputs"           # checkpoints / logs
RAW_AUDIO_DIR = DRIVE_ROOT / "dataset_clean"  # optional raw cleaned WAVs
TRANSCRIPT = DRIVE_ROOT / "si_lk.lines.txt"   # optional if preparing here
REPO = Path("/content/model-using-openSLR")

for path in (DRIVE_ROOT, DATASET_DIR, OUTPUT_DIR):
    path.mkdir(parents=True, exist_ok=True)

print("Drive root:", DRIVE_ROOT)

## Step 1 — Upload data to Google Drive

### Option A (recommended): upload the already-prepared dataset from your PC

On your Windows PC you already ran:

```powershell
python scripts/prepare_vits_dataset.py `
  --audio-dir "C:\Users\PCland\Desktop\5th sem project\dataset_clean" `
  --transcript-file "data\metadata\si_lk.lines.txt" `
  --output-dir "data\prepared\sinhala_vits"
```

Upload the folder `data/prepared/sinhala_vits` to:

`MyDrive/sinhala-vits/dataset/`

It must contain:

```text
dataset/
  wavs/*.wav
  metadata.csv
  metadata_train.csv
  metadata_val.csv
  preparation_report.json
```

### Option B: upload raw cleaned WAVs + transcript and prepare in Colab

Upload:

- all cleaned WAVs → `MyDrive/sinhala-vits/dataset_clean/`
- `si_lk.lines.txt` → `MyDrive/sinhala-vits/si_lk.lines.txt`

## Step 2 — Clone this repo and install Coqui TTS

In [ ]:
import os
import subprocess
import sys

REPO_URL = os.environ.get(
    "SINHALA_VITS_REPO",
    "https://github.com/DSEgrp18/model-using-openSLR.git",
)

if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)

# Old `pip install TTS` breaks on Colab Python 3.12.
# Use the maintained fork: coqui-tts (imports stay as TTS.*).
# transformers>=4.57 provides isin_mps_friendly required by coqui-tts.
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "TTS", "trainer"],
    check=False,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-U",
        "coqui-tts[notebooks]",
        "transformers>=4.57.0",
        "librosa",
        "soundfile",
        "tqdm",
    ],
    check=True,
)

sys.path.insert(0, str(REPO / "src"))
os.chdir(REPO)

import TTS
from trainer import Trainer  # noqa: F401
import transformers

print("Repo:", REPO)
print("TTS version:", getattr(TTS, "__version__", "unknown"))
print("transformers:", transformers.__version__)
print("Install OK")

## Step 3 — Prepare dataset in Colab (only if you used Option B)

In [ ]:
import subprocess
import sys

need_prepare = not (DATASET_DIR / "metadata_train.csv").exists()
if need_prepare:
    assert RAW_AUDIO_DIR.exists(), f"Upload cleaned WAVs to {RAW_AUDIO_DIR}"
    assert TRANSCRIPT.exists(), f"Upload transcript to {TRANSCRIPT}"
    subprocess.run([
        sys.executable, "scripts/prepare_vits_dataset.py",
        "--audio-dir", str(RAW_AUDIO_DIR),
        "--transcript-file", str(TRANSCRIPT),
        "--output-dir", str(DATASET_DIR),
        "--sample-rate", "22050",
        "--seed", "42",
    ], check=True)
else:
    print("Prepared dataset already found at", DATASET_DIR)

!wc -l "{DATASET_DIR}/metadata_train.csv" "{DATASET_DIR}/metadata_val.csv"
!ls "{DATASET_DIR}/wavs" | head

## Step 4 — Smoke-check metadata and speakers

In [ ]:
from collections import Counter
from pathlib import Path

train_rows = (DATASET_DIR / "metadata_train.csv").read_text(encoding="utf-8").splitlines()
val_rows = (DATASET_DIR / "metadata_val.csv").read_text(encoding="utf-8").splitlines()
speakers = Counter(line.split("|")[2] for line in train_rows if line.strip())
missing = [
    line.split("|")[0]
    for line in train_rows + val_rows
    if line.strip() and not (DATASET_DIR / "wavs" / f"{line.split('|')[0]}.wav").exists()
]
print(f"train={len(train_rows)} val={len(val_rows)} speakers={len(speakers)}")
print("speakers:", dict(speakers))
assert not missing, f"Missing wavs: {missing[:10]}"
print("Dataset check passed")

## Step 5 — Train VITS

### Recommended settings

| GPU | batch_size | notes |
|-----|------------|-------|
| T4 16GB | 8–12 | safer default |
| L4 24GB | 16 | good default |
| A100 40GB | 24–32 | faster |

Training 1000 epochs on ~1100 clips usually takes **many hours**. Keep Colab awake and save to Drive so you can resume.

### Resume later

If Colab disconnects, set `CONTINUE_PATH` to the latest run folder under `outputs/` (the folder that contains `config.json` and checkpoints).

In [ ]:
import subprocess
import sys

BATCH_SIZE = 12          # lower to 8 if you hit OOM on T4
EPOCHS = 1000
CONTINUE_PATH = ""       # e.g. "/content/drive/MyDrive/sinhala-vits/outputs/sinhala_vits_openslr30-September-..."

cmd = [
    sys.executable, "scripts/train_vits.py",
    "--dataset-path", str(DATASET_DIR),
    "--output-path", str(OUTPUT_DIR),
    "--run-name", "sinhala_vits_openslr30",
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--eval-batch-size", "4",
    "--num-loader-workers", "2",
    "--sample-rate", "22050",
]
if CONTINUE_PATH:
    cmd += ["--continue-path", CONTINUE_PATH]

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

## Step 6 — Find the best checkpoint

After training (or during), list run folders and pick a `best_model*.pth` or latest checkpoint.

In [ ]:
from pathlib import Path

runs = sorted(OUTPUT_DIR.glob("sinhala_vits_openslr30*"))
assert runs, f"No runs found in {OUTPUT_DIR}"
RUN_DIR = runs[-1]
print("Using run:", RUN_DIR)

checkpoints = sorted(RUN_DIR.glob("*.pth"))
for path in checkpoints[-10:]:
    print(path.name, path.stat().st_size // (1024 * 1024), "MB")

CONFIG_PATH = RUN_DIR / "config.json"
MODEL_PATH = next(RUN_DIR.glob("best_model*.pth"), checkpoints[-1] if checkpoints else None)
assert MODEL_PATH and CONFIG_PATH.exists(), "Checkpoint/config missing"
print("MODEL:", MODEL_PATH)
print("CONFIG:", CONFIG_PATH)

## Step 7 — Synthesize Sinhala speech

In [ ]:
from IPython.display import Audio, display
from TTS.utils.synthesizer import Synthesizer

TEXT = "ආයුබෝවන්, මම සිංහලෙන් කතා කරනවා."
SPEAKER = "sin_2241"  # one of: sin_2241, sin_2282, sin_3531, ...
OUT_WAV = OUTPUT_DIR / "samples" / f"{SPEAKER}_demo.wav"
OUT_WAV.parent.mkdir(parents=True, exist_ok=True)

synth = Synthesizer(
    tts_checkpoint=str(MODEL_PATH),
    tts_config_path=str(CONFIG_PATH),
    use_cuda=True,
)
wav = synth.tts(text=TEXT, speaker_name=SPEAKER)
synth.save_wav(wav, str(OUT_WAV))
print("Wrote", OUT_WAV)
display(Audio(str(OUT_WAV)))

## Step 8 — Optional CLI inference

You can also synthesize from a shell cell:

In [ ]:
!python scripts/infer_vits.py \
  --model-path "{MODEL_PATH}" \
  --config-path "{CONFIG_PATH}" \
  --text "ඔබට සුභ දවසක් වේවා" \
  --speaker sin_4191 \
  --output "{OUTPUT_DIR}/samples/sin_4191_hello.wav"

## Troubleshooting

### CUDA out of memory
- Lower `BATCH_SIZE` to `8` or `4`.
- Restart runtime and run from Step 2 again.
- Keep `max_audio_len` at 12 seconds (already set in the trainer).

### Colab disconnected
- Re-mount Drive.
- Set `CONTINUE_PATH` to the previous run directory.
- Re-run training.

### Garbled / robotic speech early on
- Normal for the first few tens of thousands of steps.
- Keep training; listen to samples after ~100–300 epochs.

### Wrong speaker / weak speaker similarity
- Confirm you pass a valid speaker id from training metadata.
- Multi-speaker quality is limited by ~100 utterances per speaker.

### Unmatched audio count is high
- Expected with the current official transcript file.
- Only matched transcript↔WAV pairs are trained.

## License reminder

OpenSLR Resource 30 is **CC BY-SA 4.0** (Google). Keep attribution if you redistribute data or derivatives.